# v0 추천 결과 보기

`algo/content_v0.py`를 불러와 샘플 유저 6명의 추천을 표로 본다. 코드는 `.py`가 원본이고, 이 노트북은 결과 확인용이다.

> 샘플 데이터는 가짜다 (`scripts/make_sample_data.py`로 생성).

In [ ]:
import sys, pathlib
# 저장소 루트를 경로에 넣는다 (notebooks 폴더에서 실행해도 import 되게)
ROOT = pathlib.Path.cwd()
while not (ROOT / 'model').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from datetime import datetime, timezone
from model.algo import content_v0 as v0

entities, contents, users = v0.load_sample()
NOW = datetime(2026, 10, 14, 12, 0, tzinfo=timezone.utc)
print(f'작품 {len(entities)} · 콘텐츠 {len(contents)} · 유저 {len(users)}')

## 유저별 추천 상위 10개

`점수 내역`은 이유별 점수다. 어떤 이유가 순위를 끌어올렸는지 볼 수 있다.

In [ ]:
def show(user):
    print(f"■ {user['name']}  유형 {user['content_types']} · 장르 {user['genres']} · 작품 {[entities[w]['name'] for w in user['favorite_works']]}")
    for rank, s in enumerate(v0.recommend(user, contents, entities, NOW), 1):
        c = s.content
        parts = ', '.join(f'{k} {v:.2f}' for k, v in s.parts.items())
        flag = ' [스포일러 흐림]' if s.blurred else ''
        print(f"  {rank:>2}. {s.score:4.2f} [{c['content_type']:<8}] {c['title']}{flag}\n        점수 내역: {parts}")
    print()

for u in users:
    show(u)

## 가중치 바꿔 보기

아래 값을 바꾸고 다시 실행하면 순위가 어떻게 바뀌는지 바로 볼 수 있다. 예: 작품 가중치를 낮추면 좋아하는 작품 밖의 글이 올라온다.

In [ ]:
v0.W_WORK = 0.8   # 기본 1.5
v0.W_TYPE = 1.5   # 기본 1.0
show(next(u for u in users if u['user_id'] == 'yeonju'))